In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q lightgbm

In [ ]:
import lightgbm as lgb
print("LightGBM version:", lgb.__version__)

LightGBM version: 4.6.0


In [ ]:
import pandas as pd
import numpy as np

# Load the fixed E1 splits
train = pd.read_csv("/content/drive/MyDrive/Fraud_Project/E1_train.csv")
validation = pd.read_csv("/content/drive/MyDrive/Fraud_Project/E1_validation.csv")
test = pd.read_csv("/content/drive/MyDrive/Fraud_Project/E1_test.csv")

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (18734, 23)
Validation: (6322, 23)
Test: (6185, 23)


In [ ]:
def create_financial_features(df):
    df = df.copy()

    # Avoid division by zero
    df["current_ratio"] = (
        df["AssetsCurrent"] / df["LiabilitiesCurrent"].replace(0, np.nan)
    )

    df["cash_ratio"] = (
        df["CashAndCashEquivalentsAtCarryingValue"]
        / df["LiabilitiesCurrent"].replace(0, np.nan)
    )

    df["current_assets_ratio"] = (
        df["AssetsCurrent"] / df["Assets"].replace(0, np.nan)
    )

    df["liabilities_to_assets"] = (
        df["Liabilities"] / df["Assets"].replace(0, np.nan)
    )

    df["debt_to_equity"] = (
        df["Liabilities"] / df["StockholdersEquity"].replace(0, np.nan)
    )

    df["roa"] = (
        df["NetIncomeLoss"] / df["Assets"].replace(0, np.nan)
    )

    df["roe"] = (
        df["NetIncomeLoss"] / df["StockholdersEquity"].replace(0, np.nan)
    )

    df["operating_return_on_assets"] = (
        df["OperatingIncomeLoss"] / df["Assets"].replace(0, np.nan)
    )

    return df


train = create_financial_features(train)
validation = create_financial_features(validation)
test = create_financial_features(test)

In [ ]:
financial_features = [
    "price",
    "Assets",
    "AssetsCurrent",
    "CashAndCashEquivalentsAtCarryingValue",
    "CommonStockSharesIssued",
    "CommonStockValue",
    "IncomeTaxExpenseBenefit",
    "LiabilitiesCurrent",
    "OperatingIncomeLoss",
    "StockholdersEquity",
    "NetIncomeLoss",
    "Liabilities",
    "current_ratio",
    "cash_ratio",
    "current_assets_ratio",
    "liabilities_to_assets",
    "debt_to_equity",
    "roa",
    "roe",
    "operating_return_on_assets"
]

X_train = train[financial_features].copy()
X_val = validation[financial_features].copy()
X_test = test[financial_features].copy()

y_train = train["is_fraud"].copy()
y_val = validation["is_fraud"].copy()
y_test = test["is_fraud"].copy()

print("Number of features:", len(financial_features))
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

Number of features: 20
X_train: (18734, 20)
X_val: (6322, 20)
X_test: (6185, 20)


In [ ]:
from sklearn.impute import SimpleImputer

# Replace infinite values with NaN
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_val = X_val.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

# Fit imputer ONLY on training data
imputer = SimpleImputer(strategy="median")

X_train_imp = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=financial_features,
    index=X_train.index
)

X_val_imp = pd.DataFrame(
    imputer.transform(X_val),
    columns=financial_features,
    index=X_val.index
)

X_test_imp = pd.DataFrame(
    imputer.transform(X_test),
    columns=financial_features,
    index=X_test.index
)

print("Missing values after imputation:")
print("Train:", X_train_imp.isna().sum().sum())
print("Validation:", X_val_imp.isna().sum().sum())
print("Test:", X_test_imp.isna().sum().sum())

Missing values after imputation:
Train: 0
Validation: 0
Test: 0


In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

print("Training negatives:", (y_train == 0).sum())
print("Training positives:", (y_train == 1).sum())
print("scale_pos_weight:", scale_pos_weight)

Training negatives: 18579
Training positives: 155
scale_pos_weight: 119.86451612903225


In [ ]:
from lightgbm import LGBMClassifier

lgbm_model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary",
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lgbm_model.fit(
    X_train_imp,
    y_train,
    eval_set=[(X_val_imp, y_val)],
    eval_metric="auc"
)

print("LightGBM training completed.")

LightGBM training completed.


In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# Validation probabilities
y_val_prob_lgbm = lgbm_model.predict_proba(X_val_imp)[:, 1]

# Default threshold = 0.5
y_val_pred_lgbm = (y_val_prob_lgbm >= 0.5).astype(int)

print("LightGBM Validation Results")
print("-" * 40)

print("Precision :", precision_score(y_val, y_val_pred_lgbm, zero_division=0))
print("Recall    :", recall_score(y_val, y_val_pred_lgbm, zero_division=0))
print("F1-score  :", f1_score(y_val, y_val_pred_lgbm, zero_division=0))
print("ROC-AUC   :", roc_auc_score(y_val, y_val_prob_lgbm))
print("PR-AUC    :", average_precision_score(y_val, y_val_prob_lgbm))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_lgbm))

LightGBM Validation Results
----------------------------------------
Precision : 0.0
Recall    : 0.0
F1-score  : 0.0
ROC-AUC   : 0.48381288416572155
PR-AUC    : 0.008568454122918413

Confusion Matrix:
[[6259    6]
 [  57    0]]


In [ ]:
thresholds = [0.50, 0.30, 0.20, 0.10, 0.05, 0.02, 0.01, 0.005]

print("LightGBM Threshold Analysis")
print("-" * 75)
print(f"{'Threshold':<12}{'Predicted Fraud':<18}{'Precision':<15}{'Recall':<15}{'F1':<10}")

for threshold in thresholds:
    y_pred = (y_val_prob_lgbm >= threshold).astype(int)

    precision = precision_score(y_val, y_pred, zero_division=0)
    recall = recall_score(y_val, y_pred, zero_division=0)
    f1 = f1_score(y_val, y_pred, zero_division=0)

    print(
        f"{threshold:<12}"
        f"{y_pred.sum():<18}"
        f"{precision:<15.4f}"
        f"{recall:<15.4f}"
        f"{f1:<10.4f}"
    )

LightGBM Threshold Analysis
---------------------------------------------------------------------------
Threshold   Predicted Fraud   Precision      Recall         F1        
0.5         6                 0.0000         0.0000         0.0000    
0.3         9                 0.0000         0.0000         0.0000    
0.2         10                0.0000         0.0000         0.0000    
0.1         15                0.0000         0.0000         0.0000    
0.05        26                0.0000         0.0000         0.0000    
0.02        54                0.0000         0.0000         0.0000    
0.01        80                0.0000         0.0000         0.0000    
0.005       118               0.0000         0.0000         0.0000    


In [ ]:
# LightGBM controlled tuning experiment

from lightgbm import LGBMClassifier

lgbm_configs = {
    "LGBM_A": {
        "n_estimators": 300,
        "learning_rate": 0.05,
        "num_leaves": 15,
        "max_depth": -1
    },

    "LGBM_B": {
        "n_estimators": 500,
        "learning_rate": 0.03,
        "num_leaves": 31,
        "max_depth": -1
    },

    "LGBM_C": {
        "n_estimators": 500,
        "learning_rate": 0.05,
        "num_leaves": 63,
        "max_depth": -1
    },

    "LGBM_D": {
        "n_estimators": 500,
        "learning_rate": 0.05,
        "num_leaves": 31,
        "max_depth": 5
    }
}

results = []

for name, params in lgbm_configs.items():

    model = LGBMClassifier(
        **params,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary",
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    model.fit(
        X_train_imp,
        y_train
    )

    val_prob = model.predict_proba(X_val_imp)[:, 1]

    results.append({
        "Model": name,
        "ROC-AUC": roc_auc_score(y_val, val_prob),
        "PR-AUC": average_precision_score(y_val, val_prob)
    })

results_df = pd.DataFrame(results).sort_values(
    "PR-AUC",
    ascending=False
)

print(results_df.to_string(index=False))

 Model  ROC-AUC   PR-AUC
LGBM_A 0.516588 0.010490
LGBM_B 0.517184 0.009497
LGBM_D 0.431142 0.007723
LGBM_C 0.403534 0.007425


In [ ]:
final_lgbm_e1 = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=15,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary",
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

final_lgbm_e1.fit(
    X_train_imp,
    y_train
)

print("Selected E1 LightGBM model trained.")

Selected E1 LightGBM model trained.


In [ ]:
y_val_prob_final_lgbm = final_lgbm_e1.predict_proba(X_val_imp)[:, 1]

print("Final E1 LightGBM validation metrics")
print("-" * 45)
print("ROC-AUC:",
      roc_auc_score(y_val, y_val_prob_final_lgbm))
print("PR-AUC:",
      average_precision_score(y_val, y_val_prob_final_lgbm))

Final E1 LightGBM validation metrics
---------------------------------------------
ROC-AUC: 0.5165875582811779
PR-AUC: 0.010489783478611165


In [ ]:
 import joblib
 joblib.dump(
    final_lgbm_e1,
    "/content/drive/MyDrive/Fraud_Project/E1_LightGBM_model.pkl"
)

print("LightGBM model saved.")

LightGBM model saved.


In [ ]:
lgbm_threshold = 0.5

y_test_prob_lgbm = final_lgbm_e1.predict_proba(X_test_imp)[:, 1]

y_test_pred_lgbm = (
    y_test_prob_lgbm >= lgbm_threshold
).astype(int)

print("E1 LightGBM — FINAL TEST RESULTS")
print("-" * 45)

print("Precision :",
      precision_score(y_test, y_test_pred_lgbm, zero_division=0))

print("Recall    :",
      recall_score(y_test, y_test_pred_lgbm, zero_division=0))

print("F1-score  :",
      f1_score(y_test, y_test_pred_lgbm, zero_division=0))

print("ROC-AUC   :",
      roc_auc_score(y_test, y_test_prob_lgbm))

print("PR-AUC    :",
      average_precision_score(y_test, y_test_prob_lgbm))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred_lgbm))

E1 LightGBM — FINAL TEST RESULTS
---------------------------------------------
Precision : 0.0
Recall    : 0.0
F1-score  : 0.0
ROC-AUC   : 0.5301574497631106
PR-AUC    : 0.01257799811263242

Confusion Matrix:
[[6084   37]
 [  64    0]]


In [ ]:
# LightGBM validation probability distribution

print("LightGBM Validation Probability Statistics")
print("-" * 50)

print("Minimum :", y_val_prob_lgbm.min())
print("Maximum :", y_val_prob_lgbm.max())
print("Mean    :", y_val_prob_lgbm.mean())
print("Median  :", np.median(y_val_prob_lgbm))

print("\nFraud cases:")
print(
    pd.Series(
        y_val_prob_lgbm[y_val.values == 1]
    ).describe()
)

print("\nNon-fraud cases:")
print(
    pd.Series(
        y_val_prob_lgbm[y_val.values == 0]
    ).describe()
)

LightGBM Validation Probability Statistics
--------------------------------------------------
Minimum : 1.8304607374507212e-11
Maximum : 0.827054858186604
Mean    : 0.0014425703691231205
Median  : 1.2838800699925834e-06

Fraud cases:
count    5.700000e+01
mean     7.707009e-05
std      2.764747e-04
min      2.335740e-10
25%      1.248990e-07
50%      8.531170e-07
75%      9.258691e-06
max      1.656263e-03
dtype: float64

Non-fraud cases:
count    6.265000e+03
mean     1.454994e-03
std      2.362294e-02
min      1.830461e-11
25%      1.211823e-07
50%      1.288440e-06
75%      1.421771e-05
max      8.270549e-01
dtype: float64


In [ ]:
thresholds_lgbm = [
    0.005,
    0.002,
    0.001,
    0.0005,
    0.0002,
    0.0001,
    0.00005,
    0.00001
]

print("LightGBM Extended Threshold Analysis")
print("-" * 75)

print(
    f"{'Threshold':<12}"
    f"{'Predicted Fraud':<18}"
    f"{'Precision':<15}"
    f"{'Recall':<15}"
    f"{'F1':<10}"
)

for threshold in thresholds_lgbm:

    y_pred = (
        y_val_prob_lgbm >= threshold
    ).astype(int)

    precision = precision_score(
        y_val, y_pred, zero_division=0
    )

    recall = recall_score(
        y_val, y_pred, zero_division=0
    )

    f1 = f1_score(
        y_val, y_pred, zero_division=0
    )

    print(
        f"{threshold:<12}"
        f"{y_pred.sum():<18}"
        f"{precision:<15.4f}"
        f"{recall:<15.4f}"
        f"{f1:<10.4f}"
    )

LightGBM Extended Threshold Analysis
---------------------------------------------------------------------------
Threshold   Predicted Fraud   Precision      Recall         F1        
0.005       118               0.0000         0.0000         0.0000    
0.002       195               0.0000         0.0000         0.0000    
0.001       295               0.0068         0.0351         0.0114    
0.0005      405               0.0099         0.0702         0.0173    
0.0002      613               0.0082         0.0877         0.0149    
0.0001      790               0.0063         0.0877         0.0118    
5e-05       1044              0.0067         0.1228         0.0127    
1e-05       1773              0.0079         0.2456         0.0153    


In [ ]:
lgbm_threshold = 0.0005

y_test_prob_lgbm = lgbm_model.predict_proba(X_test_imp)[:, 1]

y_test_pred_lgbm = (
    y_test_prob_lgbm >= lgbm_threshold
).astype(int)

print("E1 LightGBM — FINAL TEST RESULTS")
print("-" * 45)

print("Selected threshold:", lgbm_threshold)

print("Precision :",
      precision_score(
          y_test,
          y_test_pred_lgbm,
          zero_division=0
      ))

print("Recall    :",
      recall_score(
          y_test,
          y_test_pred_lgbm,
          zero_division=0
      ))

print("F1-score  :",
      f1_score(
          y_test,
          y_test_pred_lgbm,
          zero_division=0
      ))

print("ROC-AUC   :",
      roc_auc_score(
          y_test,
          y_test_prob_lgbm
      ))

print("PR-AUC    :",
      average_precision_score(
          y_test,
          y_test_prob_lgbm
      ))

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        y_test_pred_lgbm
    )
)

E1 LightGBM — FINAL TEST RESULTS
---------------------------------------------
Selected threshold: 0.0005
Precision : 0.013888888888888888
Recall    : 0.078125
F1-score  : 0.02358490566037736
ROC-AUC   : 0.5264152099330175
PR-AUC    : 0.011810912188935284

Confusion Matrix:
[[5766  355]
 [  59    5]]
